
# Lab 4: Feature Engineering & Correlation Analysis
**Student Name:** Shaheer Khan  
**Registration No:** 22JZELE0457  

This notebook covers:
1. **Feature Extraction** – creating temporal features (hour, month, weekday, week, day of year, quarter, weekend, day/night, and seasons).
2. **Correlation Analysis** – computing Pearson, Kendall, and Spearman correlations between the target variable (`aep`) and all features.



## Part 1: Feature Extraction

### Objective
Extract meaningful temporal features from the datetime column to improve model performance.

### Steps
1. Load the cleaned dataset (with holiday indicator).
2. Extract `Hour`, `Month`, `Day_Of_Week`, `Week`, `yearday`, `quarter`.
3. Create binary features: `weekend` and `day_night`.
4. Create season indicators: `winter`, `spring`, `summer`, `fall`.
5. Reorder columns and save the final feature‑rich dataset.


In [1]:

import pandas as pd
import numpy as np

# Adjust path according to your environment
df = pd.read_csv(r'C:\Users\PMYLS\Documents\MachineLearningLaB\4_AEP_Introducing_holidays.csv', parse_dates=['Datetime'])
print("Data shape:", df.shape)
df.head()


Data shape: (121296, 4)


,Datetime,AEP_MW,Date,Holiday
0,2004-10-01 01:00:00,12379.0,2004-10-01,0
1,2004-10-01 02:00:00,11935.0,2004-10-01,0
2,2004-10-01 03:00:00,11692.0,2004-10-01,0
3,2004-10-01 04:00:00,11597.0,2004-10-01,0
4,2004-10-01 05:00:00,11681.0,2004-10-01,0


In [2]:

# Extract temporal features
df['Hour'] = df['Datetime'].dt.hour
df['Month'] = df['Datetime'].dt.month
df['Day_Of_Week'] = df['Datetime'].dt.weekday          # Monday=0, Sunday=6
df['Week'] = df['Datetime'].dt.isocalendar().week
df['yearday'] = df['Datetime'].dt.dayofyear
df['quarter'] = df['Datetime'].dt.quarter

# Weekend indicator (Saturday=5, Sunday=6)
df['weekend'] = (df['Day_Of_Week'] >= 5).astype(int)

# Day/night indicator (8:00–16:00 considered day)
df['day_night'] = ((df['Hour'] >= 8) & (df['Hour'] <= 16)).astype(int)

# Season indicators (Northern Hemisphere)
df['winter'] = ((df['Month'] == 12) | (df['Month'] == 1) | (df['Month'] == 2)).astype(int)
df['spring'] = ((df['Month'] == 3) | (df['Month'] == 4) | (df['Month'] == 5)).astype(int)
df['summer'] = ((df['Month'] == 6) | (df['Month'] == 7) | (df['Month'] == 8)).astype(int)
df['fall']   = ((df['Month'] == 9) | (df['Month'] == 10) | (df['Month'] == 11)).astype(int)

print("Features added. Shape:", df.shape)
df.head()


Features added. Shape: (121296, 16)


,Datetime,AEP_MW,Date,Holiday,Hour,Month,Day_Of_Week,Week,yearday,quarter,weekend,day_night,winter,spring,summer,fall
0,2004-10-01 01:00:00,12379.0,2004-10-01,0,1,10,4,40,275,4,0,0,0,0,0,1
1,2004-10-01 02:00:00,11935.0,2004-10-01,0,2,10,4,40,275,4,0,0,0,0,0,1
2,2004-10-01 03:00:00,11692.0,2004-10-01,0,3,10,4,40,275,4,0,0,0,0,0,1
3,2004-10-01 04:00:00,11597.0,2004-10-01,0,4,10,4,40,275,4,0,0,0,0,0,1
4,2004-10-01 05:00:00,11681.0,2004-10-01,0,5,10,4,40,275,4,0,0,0,0,0,1


In [3]:

# Rename columns for consistency
df.rename(columns={
    'AEP_MW': 'aep',
    'Holiday': 'holiday',
    'Hour': 'hour',
    'Month': 'month',
    'Day_Of_Week': 'day_of_week',
    'Week': 'week_no',
    'yearday': 'year_day',
    'quarter': 'quarter',
    'day_night': 'day_night',
    'weekend': 'weekend'
}, inplace=True)

# Reorder columns for better readability
df = df.reindex(columns=[
    'Datetime', 'aep', 'year_day', 'holiday', 'weekend',
    'winter', 'spring', 'summer', 'fall',
    'hour', 'month', 'day_of_week'
])

df.head()


,Datetime,aep,year_day,holiday,weekend,winter,spring,summer,fall,hour,month,day_of_week
0,2004-10-01 01:00:00,12379.0,275,0,0,0,0,0,1,1,10,4
1,2004-10-01 02:00:00,11935.0,275,0,0,0,0,0,1,2,10,4
2,2004-10-01 03:00:00,11692.0,275,0,0,0,0,0,1,3,10,4
3,2004-10-01 04:00:00,11597.0,275,0,0,0,0,0,1,4,10,4
4,2004-10-01 05:00:00,11681.0,275,0,0,0,0,0,1,5,10,4


In [4]:

# Check for missing values
print("Missing values per column:\n", df.isnull().sum())

# Save the feature‑engineered dataset
df.to_csv(r'C:\Users\PMYLS\Documents\MachineLearningLaB\5_features_extracted.csv', index=False)
print("Feature dataset saved.")


Missing values per column:
 Datetime       0
aep            0
year_day       0
holiday        0
weekend        0
winter         0
spring         0
summer         0
fall           0
hour           0
month          0
day_of_week    0
dtype: int64
Feature dataset saved.



## Part 2: Correlation Analysis

### Objective
Compute correlation coefficients between the target variable `aep` and all features to identify which features are most linearly associated with energy consumption.

We use three methods:
- **Pearson** – linear correlation.
- **Kendall** – ordinal association.
- **Spearman** – monotonic relationship.

### Steps
1. Load the feature‑engineered dataset.
2. Compute correlations with `aep` using `corrwith()`.
3. Display results for each method.


In [5]:

# Load the feature dataset
df = pd.read_csv(r'C:\Users\PMYLS\Documents\MachineLearningLaB\5_features_extracted.csv', index_col=['Datetime'], parse_dates=['Datetime'])
print("Data shape:", df.shape)
df.head()


Data shape: (121296, 11)


,aep,year_day,holiday,weekend,winter,spring,summer,fall,hour,month,day_of_week
Datetime,,,,,,,,,,,
2004-10-01 01:00:00,12379.0,275,0,0,0,0,0,1,1,10,4
2004-10-01 02:00:00,11935.0,275,0,0,0,0,0,1,2,10,4
2004-10-01 03:00:00,11692.0,275,0,0,0,0,0,1,3,10,4
2004-10-01 04:00:00,11597.0,275,0,0,0,0,0,1,4,10,4
2004-10-01 05:00:00,11681.0,275,0,0,0,0,0,1,5,10,4


In [6]:

# Pearson correlation
print("Pearson correlation with aep:")
print(df.corrwith(df["aep"], method="pearson"))


Pearson correlation with aep:
aep            1.000000
year_day      -0.124617
holiday       -0.053212
weekend       -0.267287
winter         0.328332
spring        -0.246582
summer         0.139793
fall          -0.220913
hour           0.421008
month         -0.125896
day_of_week   -0.220016
dtype: float64


In [7]:

# Kendall correlation
print("Kendall correlation with aep:")
print(df.corrwith(df["aep"], method="kendall"))


Kendall correlation with aep:
aep            1.000000
year_day      -0.082592
holiday       -0.041101
weekend       -0.218066
winter         0.278561
spring        -0.197749
summer         0.098140
fall          -0.178467
hour           0.292545
month         -0.085584
day_of_week   -0.158129
dtype: float64


In [8]:

# Spearman correlation
print("Spearman correlation with aep:")
print(df.corrwith(df["aep"], method="spearman"))


Spearman correlation with aep:
aep            1.000000
year_day      -0.123249
holiday       -0.050335
weekend       -0.267060
winter         0.341147
spring        -0.242178
summer         0.120189
fall          -0.218564
hour           0.431821
month         -0.123443
day_of_week   -0.219619
dtype: float64


In [9]:

# Verify no missing values
print("Missing values after feature extraction:\n", df.isnull().sum())


Missing values after feature extraction:
 aep            0
year_day       0
holiday        0
weekend        0
winter         0
spring         0
summer         0
fall           0
hour           0
month          0
day_of_week    0
dtype: int64



## Summary

- **Feature Extraction** added 11 new features derived from datetime: hour, month, day of week, week number, day of year, quarter, weekend flag, day/night flag, and four season indicators.
- **Correlation Analysis** revealed:
  - `hour` has the strongest positive correlation (Pearson ~0.42, Spearman ~0.43).
  - `weekend` and `day_of_week` show moderate negative correlations.
  - Seasonal features show moderate correlations (e.g., winter positive, spring/fall negative).

These insights help in selecting relevant features for time‑series forecasting models.

---

**Prepared by:** Shaheer Khan (22JZELE0457)  
